# E1.11 · Model risk management for AI systems

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.10 · The stakeholder map: who owns what](https://spbreed.github.io/cyber-commons/lessons/E1.10.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Take a validated model, add one tool, and show which parts of the validation are now void.

**Why a security engineer needs it.** The classical model-risk playbook silently breaks once the model can act: conceptual soundness was validated, and then the agent was granted write access nobody validated. The control it builds is: extend the SR 11-7 lineage — conceptual soundness, ongoing monitoring, independent validation — to non-deterministic, tool-using systems, and name where it still holds.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Model risk management has forty years of doctrine on validating models — conceptual soundness, ongoing monitoring, independent validation. Most of it transfers. The part that does not is the part where the model calls tools.

> **At CyberTravels.** Forty years of model-risk doctrine transfers to CyberTravels. The part that does not is the part where the model calls `issue_refund`.

## 2 · The framework

```
   SR 11-7 lineage                 what an agent adds
   +----------------------+        +-----------------------+
   | conceptual soundness |  ok    | it calls tools        |
   | ongoing monitoring   |  ok    | it acts on the world  |
   | independent validation| ok    | output is not a number|
   +----------------------+        +-----------------------+

   most of forty years of doctrine transfers. the tool call does not.
```

Model risk management is not new. The SR 11-7 lineage has governed models in
regulated institutions for over a decade, and its three pillars are sound:

1. **Conceptual soundness** — is the method appropriate for the purpose?
2. **Ongoing monitoring** — is it still performing as validated?
3. **Independent validation** — did someone other than the builder check?

All three still hold for AI systems. What breaks is not the framework but a
silent assumption underneath it: **that a model produces an output, and a human
decides what to do with it.**

Once the model can call a tool, that assumption is void. Validation scoped to
the model's *predictions* says nothing about the model's *actions*. You can hold
a perfectly valid validation report for a system that has since been granted
write access to a production database, and nothing in the classical process is
required to notice.

So the extension is narrow and specific: the unit of validation becomes the
**model plus its tool surface plus its autonomy level**, and any change to any of
the three triggers revalidation — not just a change to the weights.

## 3 · The procedure, as a skill

A model validated with no tools at L1 is deployed with three tools at L3 — same model, same version, different system. The skill diffs validated against deployed and lists what the monitoring never observes.

### The skill — [`skills/grc/model-risk-validation-scope/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/model-risk-validation-scope/SKILL.md)

```yaml
name: model-risk-validation-scope
description: >-
  Check whether a model validated under one autonomy level and tool set is still
  covered by that validation as deployed, and find the monitoring that stops at
  the model's output. Use when applying model risk management to an agentic
  system.
allowed-tools: Read, Grep, Glob
```

# The validation was of a system that had no tools

Model risk management carries three pillars — validation, monitoring,
governance — and each makes an assumption that agentic deployment breaks.
Validation assumes the thing validated is the thing deployed; a model validated
with no tools at L1 and deployed with three tools at L3 is the same model and a
different system, and the validation does not cover it.

## When to use this

Applying an existing model risk framework to agents, and at every autonomy or
tool-manifest change afterwards.

## Procedure

**1 — Write down what was validated.** Model, version, tool set, autonomy level,
data scope. Validation scope is usually recorded as the model alone, which is
the defect.

**2 — Write down what is deployed,** in the same fields. Then diff. Any
difference in tools or autonomy means the validation does not cover the
deployment, and that sentence is the finding.

**3 — Check what monitoring observes.** Most monitors the model's output
distribution. List what it does not observe — rows written to production, actions
taken, resources reached — because that is where agentic risk lives.

**4 — Name the re-validation triggers.** Tool added, autonomy raised, model
version changed, data scope widened. Without triggers, re-validation happens on
the audit calendar, which is the assumption that failed in the first place.

**5 — Report per pillar.** Validation coverage, monitoring blind spots,
governance triggers. Three findings with three owners rather than one finding
about the framework.

## Example

**Input** — the fixture committed at the top of [`scripts/model_risk_validation_scope.py`](scripts/model_risk_validation_scope.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
pillar                  what it asks                                  what it quietly assumes
conceptual_soundness    is the method appropriate for the purpose     assumes the purpose is stable and stated
independent_validation  did someone other than the builder check      assumes the thing checked is the thing deployed
ongoing_monitoring      is it still performing as validated           assumes performance is what changes

All three survive contact with AI. The assumptions are what break.
validation still covers what is deployed: False
   tool surface changed: ['db_update', 'read_ticket', 'write_ticket']
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "validated": {"model": "str", "version": "str", "tools": ["str"], "autonomy": "str", "data": ["str"]},
  "deployed": {"model": "str", "version": "str", "tools": ["str"], "autonomy": "str", "data": ["str"]},
  "covers": false,
  "differences": ["str"],
  "monitoring": {"observes": ["str"], "does_not_observe": ["str"]},
  "revalidation_triggers": ["str"]
}
```

## Failure modes

- **Recording validation scope as the model.** It is the system.
- **Monitoring output distribution only.** The actions are the risk.
- **Calendar re-validation.** The triggers are events, not dates.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/model-risk-validation-scope/scripts/model_risk_validation_scope.py
SCRIPT = "skills/grc/model-risk-validation-scope/scripts/model_risk_validation_scope.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The three SR 11-7 pillars, each with the assumption it quietly makes. A system validated with no tools at L1 is shown deployed with three tools at L3 — same model, same version — and the validation no longer covers it. Monitoring reports 200 clean runs of summarisation accuracy while four action-level metrics have no threshold at all, and four revalidation triggers classical MRM would miss are named.

## Your turn

Take one validated model in your estate and list the tools it holds today. If any of them post-dates the validation report, the report is describing a different system.

---

**Next → [E1.12 · Working the seams](https://spbreed.github.io/cyber-commons/lessons/E1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*